In [6]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Örnek veri (önceki ile aynı)
df = pd.DataFrame({
    'sessionID': [1, 1, 2, 2, 3],
    'createdAt_form': pd.date_range('2024-01-01', periods=5),
    'min_price': [100, 110, 200, 180, 150],
    'max_price': [200, 210, 300, 280, 250]
})

# Zamanı sayısal yap
df['timestamp'] = (df['createdAt_form'] - df['createdAt_form'].min()).dt.total_seconds() / (60 * 60 * 24)
df['avg_price'] = df[['min_price', 'max_price']].mean(axis=1)
df['price_range'] = df['max_price'] - df['min_price']

def calculate_slope_and_trend(df, value_column, threshold=0.01):
    results = []
    for session_id, group in df.groupby('sessionID'):
        group = group.sort_values('timestamp')
        if len(group) < 2:
            slope = 0
            trend = 'durağan'
        else:
            X = group['timestamp'].values.reshape(-1, 1)
            y = group[value_column].values
            model = LinearRegression().fit(X, y)
            slope = model.coef_[0]
            if slope > threshold:
                trend = 'artan'
            elif slope < -threshold:
                trend = 'azalan'
            else:
                trend = 'durağan'
        results.append({'sessionID': session_id, f'{value_column}_slope': slope, f'{value_column}_trend': trend})
    return pd.DataFrame(results)

# Fiyatlara göre trendleri hesapla
min_trends = calculate_slope_and_trend(df, 'min_price')
max_trends = calculate_slope_and_trend(df, 'max_price')
avg_trends = calculate_slope_and_trend(df, 'avg_price')
range_trends = calculate_slope_and_trend(df, 'price_range')

# Hepsini birleştir
trend_summary = min_trends.merge(max_trends, on='sessionID') \
                          .merge(avg_trends, on='sessionID') \
                          .merge(range_trends, on='sessionID')

display(trend_summary)


,sessionID,min_price_slope,min_price_trend,max_price_slope,max_price_trend,avg_price_slope,avg_price_trend,price_range_slope,price_range_trend
0,1,10.0,artan,10.0,artan,10.0,artan,0.0,durağan
1,2,-20.0,azalan,-20.0,azalan,-20.0,azalan,0.0,durağan
2,3,0.0,durağan,0.0,durağan,0.0,durağan,0.0,durağan


In [10]:
df.groupby('sessionID').value_counts()

sessionID  createdAt_form  min_price  max_price  timestamp  avg_price  price_range
1          2024-01-01      100        200        0.0        150.0      100            1
           2024-01-02      110        210        1.0        160.0      100            1
2          2024-01-03      200        300        2.0        250.0      100            1
           2024-01-04      180        280        3.0        230.0      100            1
3          2024-01-05      150        250        4.0        200.0      100            1
Name: count, dtype: int64